# Channel Ablation Study: Scratch Baseline

В данном ноутбуке представлен анализ влияния набора ЭЭГ-каналов на качество модели в условиях обучения с нуля (scratch).

**Цель эксперимента**

Оценить, как уменьшение количества используемых каналов влияет на качество модели, и определить, возможно ли сократить число отведений без существенной потери качества.

*Рассматриваются три набора каналов:*

- ch14 — полный набор из 14 каналов (baseline)
- ch7 — сокращённый набор: Cz, Pz, Oz, P3, P4, PO7, PO8
- ch3 — минимальный набор: Cz, Pz, Oz

**Протокол эксперимента**
Модель: UNet1D Encoder + GAP + Linear (512 → 2)
Обучение: scratch (без SSL-предобучения)
Данные: BigP3BCI (benchmark subjects)
Разбиение: фиксированные splits (calib_pool / test_rest)
Калибровка: p∈{0,10,20,40,60,100}%
Нормализация: z-score по каналам, рассчитанный только на Calib_p
Оценка: на фиксированном test_rest без утечек

## 1. Импорты

In [ ]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import stage5_utils as u

import model_unet as m

## 2. Конфиги

In [ ]:
RUNTIME_CONFIG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,
}

In [ ]:
PATHS = {
    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_results"),
}

## 3. Воспроизводимость

In [ ]:
u.set_seed(RUNTIME_CONFIG["seed"])
print("Device:", RUNTIME_CONFIG["device"])

## 4. Пути

In [ ]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": PATHS["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": PATHS["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": PATHS["data_root"] / "bcicompiii-ds2",  
}

In [ ]:
GROUPS = ["train", "benchmark", "bcicomp3"]

### Автореестр субъектов

In [ ]:
# Автореестр субъектов
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [ ]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


## Конфиг для FT-сценариев и scratch на Test

In [ ]:
DEV_SUBJECTS = [
    'subj_054', 
    'subj_065', 
    'subj_090', 
    'subj_094'
]

In [ ]:
ALL_BENCHMARK_SUBJECTS = SUBJECT_REGISTRY["benchmark"]

FILTERED_BENCHMARK_SUBJECTS = [
    s for s in ALL_BENCHMARK_SUBJECTS
    if s not in DEV_SUBJECTS
]

print("Original benchmark subjects:", len(ALL_BENCHMARK_SUBJECTS))
print("After exclusion:", len(FILTERED_BENCHMARK_SUBJECTS))
print("Excluded:", DEV_SUBJECTS)

In [ ]:
FINAL_TEST_CONFIG = {
    "tag": "stage5_channel_ablation_scratch_benchmark",
    "subjects": FILTERED_BENCHMARK_SUBJECTS,
    "group": "benchmark",

    "p_list": [0, 10, 20, 40, 60, 100],

    # сценарии
    "ft_scenarios": ["full_ft", "low_lr_encoder", "partial_ft", "warmup"],
    "scratch_scenarios": ["scratch"],

    # SCRATCH параметры
    "scratch_lr": 3e-4,
    "scratch_weight_decay": 1e-4,

    # FT параметры
    "lr_encoder": 3e-5,
    "lr_head": 3e-4,
    "weight_decay": 1e-3,
    "warmup_epochs": 3,
}


In [ ]:
CHANNELS_14 = ["Fz", "Cz", "Pz", "Oz", "P3", "P4", "PO7", "PO8", "F3", "F4", "C3", "C4", "CP3", "CP4"]

CHANNEL_SETS = {
    "ch14": CHANNELS_14,
    "ch7": ["Cz", "Pz", "Oz", "P3", "P4", "PO7", "PO8"],
    "ch3": ["Cz", "Pz", "Oz"],
}

CHANNEL_SET_NAME = "ch3"
CHANNEL_NAMES = CHANNEL_SETS[CHANNEL_SET_NAME]
CHANNEL_IDX = [CHANNELS_14.index(ch) for ch in CHANNEL_NAMES]

## Пути для FT и Scratch на Test

In [ ]:
RUN_DIR = PATHS["results_root"] / "channel_ablation" / CHANNEL_SET_NAME / FINAL_TEST_CONFIG["tag"]

ARTIFACT_DIRS = {
    "root": RUN_DIR,
    "ft": RUN_DIR / "ft_strategies",
    "scratch": RUN_DIR / "scratch",
    "history": RUN_DIR / "history",
    "predictions": RUN_DIR / "predictions",
    "tables": RUN_DIR / "tables",
    "figures": RUN_DIR / "figures",
}

for path in ARTIFACT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

## Полный прогон Scratch на Test

In [ ]:
# =========================
# Full Test run: channel ablation scratch on benchmark
# =========================

print("SCRATCH DIR:", ARTIFACT_DIRS["scratch"])
print("Channel set:", CHANNEL_SET_NAME)
print("Channels:", CHANNEL_NAMES)
print("Channel idx:", CHANNEL_IDX)
print("Group:", FINAL_TEST_CONFIG["group"])
print("N benchmark subjects:", len(FINAL_TEST_CONFIG["subjects"]))
print("p_list:", FINAL_TEST_CONFIG["p_list"])
print("scratch_lr:", FINAL_TEST_CONFIG["scratch_lr"])
print("scratch_weight_decay:", FINAL_TEST_CONFIG["scratch_weight_decay"])

final_test_scratch_df = u.run_many(
    subject_list=FINAL_TEST_CONFIG["subjects"],
    p_list=FINAL_TEST_CONFIG["p_list"],
    scenario_list=["scratch"],
    group=FINAL_TEST_CONFIG["group"],
    runtime_config=RUNTIME_CONFIG,
    results_root=ARTIFACT_DIRS["scratch"],
    scratch_lr=FINAL_TEST_CONFIG["scratch_lr"],
    scratch_weight_decay=FINAL_TEST_CONFIG["scratch_weight_decay"],
    channel_set_name=CHANNEL_SET_NAME,
    channel_idx=CHANNEL_IDX,
    channel_names=CHANNEL_NAMES,
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

final_test_scratch_summary_path = ARTIFACT_DIRS["scratch"] / "summary.csv"
final_test_scratch_df.to_csv(final_test_scratch_summary_path, index=False)

print("\nChannel ablation scratch run finished.")
print("Saved summary to:", final_test_scratch_summary_path)
print("Shape:", final_test_scratch_df.shape)

display(final_test_scratch_df.head())